# bonus b - split-k reduction

### the problem, from a real profiler
cudagrad trains a cnn on an rtx 4080. nvidias profiler (nsight) said two kernels ate **95% of the gpu's time**:

| kernel | share of gpu time |
|---|---|
| `conv2d_dW_kernel` (conv weight gradient) | 67.4% |
| `conv_bias_grad_kernel` (conv bias gradient) | 27.8% |
| everything else, forward passes included | ~5% |

why? each one has very few outputs (72 or 1,152 weights, 8 or 16 biases) and the kernel used **one thread per output**, so only 1-5 thread blocks ran on a gpu with 76 processors (sms). each thread then added up to **86,528** things in a row. thats about 0.2% of what the gpu can do.

### the fix: split the K dimension
a matrix multiply $C = AB^T$ has three sizes: **M** and **N** (the output is M x N) and **K** (each output is a sum of K products). when M x N is small and K is huge, cut K into S slices:

1. every (output, slice) pair works out a **partial sum**, thats S times more independent work for the gpu
2. then **reduce**: add up the S partial sums for every output

thats **split-k**. in cutlass's profiler its the `--split_k_slices` option (with `--split_k_mode serial` or `parallel`). in cudagrad its `blockIdx.y`:

```cpp
// cudagrad: conv2d_dW_kernel (simplified)
const int p  = blockIdx.x;                  // which weight (one output of the matmul)
const int r0 = blockIdx.y * chunk;          // which slice of K this block adds up
for (int r = r0 + threadIdx.x; r < r1; r += blockDim.x)
    acc += x[...] * dOut[...];              // 256 threads each add a strided part of the slice
partial[threadIdx.x] = acc;                 // then a TREE REDUCTION in shared memory:
for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
    if (threadIdx.x < stride) partial[threadIdx.x] += partial[threadIdx.x + stride];
    __syncthreads();
}
if (threadIdx.x == 0) atomicAdd(&dW[p], partial[0]);   // add this slice's partial sum into the answer
```

same deal as the main lab: `# todo:` cells are yours, and `%load solutions/NAME.py` (run it twice) if youre stuck.

## formula sheet

- matrix multiply: $C[i][j] = \sum_{k=0}^{K-1} A[i][k]\,B[j][k]$, with $M \times N$ outputs and $K$ products added up per output
- split-k with $S$ slices: $\text{chunk} = \lceil K / S \rceil$. slice $s$ covers $k$ from $s \cdot \text{chunk}$ up to $\min(K, (s+1)\cdot\text{chunk})$
- $C = \sum_{s} C_s$ where $C_s[i][j] = \sum_{k \text{ in slice } s} A[i][k]\,B[j][k]$
- independent work goes from $M N$ jobs of $K$ products each to $M N S$ jobs of about $K / S$ products each
- tree reduction of $S$ partial sums takes $\lceil \log_2 S \rceil$ rounds ($8 \to 4 \to 2 \to 1$ is 3 rounds), and $S - 1$ adds per output in total
- cute view of the K mode: `K:1` divided by the chunk is `(chunk, S):(1, chunk)`, so slice $s$ starts at offset $s \cdot \text{chunk}$
- training on many cores is split-k over the batch: $dW = \dfrac{1}{N}\sum_{\text{workers}} \;\sum_{\text{images of that worker}} dW_\text{image}$
- floating point addition isnt perfectly associative: $(a + b) + c$ can differ from $a + (b + c)$ in the last digit
- amdahls law: if a fraction $p$ of the work can run in parallel, $n$ workers give a speed-up of $\dfrac{1}{(1 - p) + p / n}$. with $p \approx 0.87$ and $n = 8$ thats about $4.2\times$, not $8\times$

## python stuff for this notebook
- `multiprocessing` runs python in seperate processes, so they really do run at the same time on different cores
  - `mp.get_context("spawn").Pool(n)` starts n worker processes
  - `pool.map(f, jobs)` runs `f(job)` for every job on the workers and gives back the results in order
  - the function has to live in a `.py` file (`addons.py` here) becuase every worker imports it fresh. a function you define in the notebook wont work
  - everything sent to a worker and back gets copied (pickled). thats part of why its not 8x faster
- `with SplitKTrainer(...) as trainer:` runs `__enter__` at the start and `__exit__` at the end, even if something crashes. here that shuts the workers down
- `global _WORKER_MODEL` lets a function change a variable that lives outside it (each worker keeps one model around)
- `inspect.getsource(f)` gives back the code of a function as text
- `os.cpu_count()` is how many cores your computer has
- `-(-K // S)` is K / S rounded up

In [1]:
import time
from babytorch import *          # the main lab (answer key versions)
from addons import batched_im2col_layout, stack, batches
import viz, checks

### b.1 tree reduction
adding S partial sums one after another takes S − 1 steps. adding them **in pairs**, all the pairs at the same time, takes only $\log_2 S$ rounds: 8 partial sums → 4 → 2 → 1. thats the loop at the end of cudagrad's kernel.

In [2]:
# answer (same as solutions/tree_reduce.py)
def tree_reduce(partials):
    """adds up a list of same-length lists in pairs, the way a gpu thread block does it in shared memory:
        [p0, p1, p2, p3]  ->  [p0+p1, p2+p3]  ->  [p0+p1+p2+p3]       (log2(n) rounds)
    every add inside one round is independent, so a gpu does a whole round at the same time"""
    while len(partials) > 1:
        paired = [[a + b for a, b in zip(partials[i], partials[i + 1])] for i in range(0, len(partials) - 1, 2)]
        if len(partials) % 2:
            paired.append(partials[-1])   # an odd one out waits for the next round
        partials = paired
    return partials[0]

In [3]:
checks.tree_reduce(tree_reduce)

pass  tree_reduce


### b.2 slicing K, the cute way
the K mode of a matrix is a 1-D layout `K:1`. cutting it into tiles of `chunk` is cute's `zipped_divide`. you get `(chunk, S):(1, chunk)`, which means "where am i inside the slice" and "which slice". slice `s` covers offsets `s·chunk` up to `s·chunk + chunk − 1`.

In [4]:
K, chunk = 784, 98
tiles = zipped_divide(Layout(K), (chunk,))
print(tiles, "->", size(tiles[1]), "slices of", size(tiles[0]))
print("slice 3 starts at offset", tiles((0,), (3,)), "and ends at", tiles((chunk - 1,), (3,)))

((98),(8)):((1),(98)) -> 8 slices of 98
slice 3 starts at offset 294 and ends at 391


In [5]:
# answer (same as solutions/matmul_t_split_k.py)
def matmul_t_split_k(A, B, splits):
    """C = A @ B^T like matmul_t, but the K dimension (the sum) gets cut into `splits` slices.
    every slice gives a partial C (they could all run at the same time), then the partials get added up"""
    K = len(A[0])
    chunk = -(-K // splits)                       # K / splits rounded up
    partials = []
    for s in range(splits):
        lo, hi = s * chunk, min(K, (s + 1) * chunk)
        partials.append(matmul_t([a[lo:hi] for a in A], [b[lo:hi] for b in B]))
    return tree_reduce(partials)

In [6]:
checks.matmul_t_split_k(matmul_t_split_k)

pass  matmul_t_split_k - same answer as one big matmul (up to rounding in the last digit)


### b.3 split-k on a real weight gradient
the weight gradient of lenet's first conv for a batch of 16 real images: 6 x 75 outputs, and each one adds up K = 16 x 784 = 12,544 products.

In [7]:
loader = DataLoader(DATA / "train.bin", per_class=2)
images = [loader.image(k) for k in range(16)]
x = stack(images).data
L = batched_im2col_layout(16, 3, 32, 32, 5)
P = size(L[0])                                                  # 16 * 784 rows, thats the K of the weight gradient
patch_columns = rows(gather(x, transpose(L)), P)                # 75 rows, 12,544 long
random.seed(0)
dy_rows = [[random.uniform(-1, 1) for _ in range(P)] for _ in range(6)]   # a made up output gradient

t = time.perf_counter(); plain = matmul_t(dy_rows, patch_columns); t_plain = time.perf_counter() - t
for S in (1, 2, 8, 64):
    t = time.perf_counter(); split = matmul_t_split_k(dy_rows, patch_columns, S); dt = time.perf_counter() - t
    print(f"S = {S:>2}: biggest difference from plain {max(abs(a - b) for a, b in zip(split, plain)):.1e}   "
          f"({dt * 1000:.0f} ms vs {t_plain * 1000:.0f} ms)")

S =  1: biggest difference from plain 0.0e+00   (24 ms vs 24 ms)
S =  2: biggest difference from plain 2.8e-14   (36 ms vs 24 ms)
S =  8: biggest difference from plain 2.8e-14   (26 ms vs 24 ms)


S = 64: biggest difference from plain 2.8e-14   (29 ms vs 24 ms)


**the same answer, but not *exactly* the same digits.** computer addition isnt perfectly associative:

In [8]:
print((0.1 + 0.2) + 0.3, "vs", 0.1 + (0.2 + 0.3))

0.6000000000000001 vs 0.6


split-k changes the *order* of the additions, so the last digit or two can change. on a gpu, `atomicAdd` makes the order depend on which thread block finishes first, so two runs that start from *the exact same* weights can still end up with slightly different ones.

on one python core split-k isnt faster, its the same multiplies plus a reduction. the point of it is **parallelism**.

### b.4 split-k across your cpu cores
training on a batch: the weight gradient is a sum over every image in the batch (thats part of K). so give each cpu core (a *worker process*) a slice of the batch. every worker runs forward + backward on its images and gives back its **partial sum** of the gradients. the main process **reduces** the partial sums with your `tree_reduce` and takes one sgd step.

| gpu (cudagrad) | here |
|---|---|
| a thread block along `blockIdx.y` | a worker process |
| a slice of K | a slice of the batch |
| shared memory tree reduction + `atomicAdd` | `tree_reduce` in the main process |

the worker code has to live in a `.py` file (`addons.py`) so the worker processes can import it. heres what it looks like:

In [9]:
# prints the code of the two pieces that matter. read the comments in it
import inspect, addons
from addons import SplitKTrainer
print(inspect.getsource(addons.worker_partial_grads))
print(inspect.getsource(SplitKTrainer.gradients))

def worker_partial_grads(job):
    """runs inside a worker process (one "thread block"): forward + backward for its slice of the batch,
    one image at a time, WITHOUT clearing the gradients in between, so they add up into a partial sum"""
    global _WORKER_MODEL
    params, images, labels = job
    if _WORKER_MODEL is None:
        _WORKER_MODEL = lenet5()
    model = _WORKER_MODEL
    for p, data in zip(model.parameters(), params):
        p.data, p.grad = data, None
    total_loss, correct = 0.0, 0
    for image, label in zip(images, labels):
        logits = model(Tensor(image, (3, 32, 32)))
        loss = softmax_cross_entropy(logits, label)
        loss.backward()
        total_loss += loss.data[0]
        correct += max(range(len(logits.data)), key=logits.data.__getitem__) == label
    return [p.grad for p in model.parameters()], total_loss, correct

    def gradients(self, images, labels, reduce):
        """fills every weight's .grad with the batch average gradient. gives ba

In [10]:
import os
loader = DataLoader(DATA / "train.bin", shuffle=True, per_class=20)
data = [pair for _, pair in zip(range(64), loader)]                   # one batch of 64
imgs, labs = [d[0] for d in data], [d[1] for d in data]
print("cpu cores on this computer:", os.cpu_count())

timings, reference = {}, None
for workers in (1, 2, 4, 8):
    model = lenet5(seed=2)
    opt = SGD(model.parameters(), 0.0)                                # lr 0, we only want the gradients
    with SplitKTrainer(model, opt, workers) as trainer:
        trainer.gradients(imgs[:workers], labs[:workers], tree_reduce)      # wake the workers up first
        t = time.perf_counter()
        trainer.gradients(imgs, labs, tree_reduce)                          # one batch of 64
        timings[workers] = time.perf_counter() - t
        grads = [list(p.grad) for p in model.parameters()]
    reference = reference or grads
    diff = max(abs(a - b) for x, y in zip(grads, reference) for a, b in zip(x, y))
    print(f"{workers} worker(s): {timings[workers] * 1000:5.0f} ms per batch of 64   "
          f"speed-up {timings[1] / timings[workers]:.1f}x   gradient difference vs 1 worker: {diff:.0e}")
viz.line_chart({"measured": [(w, timings[1] / t) for w, t in timings.items()], "perfect": [(w, w) for w in timings]},
               "split-k across cpu cores", "workers", "speed-up")

cpu cores on this computer: 15


1 worker(s):  1028 ms per batch of 64   speed-up 1.0x   gradient difference vs 1 worker: 0e+00


2 worker(s):   525 ms per batch of 64   speed-up 2.0x   gradient difference vs 1 worker: 6e-16


4 worker(s):   280 ms per batch of 64   speed-up 3.7x   gradient difference vs 1 worker: 4e-16


8 worker(s):   246 ms per batch of 64   speed-up 4.2x   gradient difference vs 1 worker: 6e-16


talk about it: why not 8x with 8 workers? some of the work doesnt split: sending the weights to every worker, collecting 62,006 partial gradients from each one, the reduction, the sgd step. that serial part caps the speed-up (**amdahls law**, its on the formula sheet). laptops also mix fast and slow cores. gpus win becuase their "workers" share memory, so nothing gets copied.

### b.5 train with it
batches of 32 split across 4 workers (8 images each). compare the images per second with the main lab's ~45-60.

In [11]:
train_loader = DataLoader(DATA / "train.bin", shuffle=True, per_class=350)
model = lenet5(seed=0)
EPOCHS, BATCH, WORKERS = 3, 32, 4
opt = SGD(model.parameters(), lr_max=0.08, lr_min=1e-5,
          total_steps=EPOCHS * -(-len(train_loader) // BATCH), schedule=cosine_annealing_lr)
start = time.time()
with SplitKTrainer(model, opt, WORKERS) as trainer:
    for epoch in range(EPOCHS):
        seen = right = 0
        for images, labels in batches(train_loader, BATCH):
            loss, correct = trainer.step(images, labels, reduce=tree_reduce)
            seen += len(images); right += correct
        print(f"epoch {epoch + 1}: train acc {right / seen:.1%}   {seen * (epoch + 1) / (time.time() - start):.0f} images/s")
acc, _ = evaluate(model, DataLoader(DATA / "test.bin"))
print(f"real test accuracy after {EPOCHS} epochs on {len(train_loader)} images: {acc:.1%}")

epoch 1: train acc 26.9%   97 images/s


epoch 2: train acc 54.1%   95 images/s


epoch 3: train acc 63.3%   83 images/s


real test accuracy after 3 epochs on 3500 images: 73.3%


### where to go from here
- **cutlass** has split-k matrix multiplies and convolutions built in. its convolutions are "implicit gemm": the im2col matrix never gets built, every tile of it gets formed on the fly while its loaded into shared memory. thats your `im2col_layout`, worked out by gpu threads
- **cute dsl** lets you write those kernels in python: `pip install nvidia-cutlass-dsl` (needs an nvidia gpu, ampere or newer, on linux). layouts look just like ours: `cute.make_layout((3, 32, 32), stride=(1024, 32, 1))`
- **cudagrad** has the split-k kernels and the nsight profiles from the top of this notebook: https://github.com/notYnoj/cudagrad